# 05. 계정 만들기 — 학과장 · 서버 관리자 · 학생

| 항목 | 내용 |
|---|---|
| 목표 | 학과장 계정을 만들고 관리자 가입·승인, 학생 가입까지 확인 |
| 소요 시간 | 30분 |
| 시작 전 조건 | 04 완료(사이트 접속 가능), 인증 앱 |
| 실행 위치 | 🖥 서버 콘솔 (절마다 다시 표시) |

> 대문자 대괄호 `[LIKE_THIS]` 는 **자기 환경의 값으로 바꿔 쓰는 자리표시자**입니다. 표는 [00_overview](00_overview.ipynb)에 있습니다.
> 이 노트북의 셀은 **문서**입니다. 설명대로 해당 위치(서버 콘솔 등)에서 직접 실행하세요. 명령에 따라 서버 설정이 바뀝니다.

## A. 학과장 계정 (🖥 서버 콘솔, 최초 1회)

학과장은 사이트에서 가입할 수 없고 명령으로 만듭니다. **비밀번호는 입력해도 화면에 안 보입니다**(두 번 입력).
- 아이디: 영소문자로 시작, 영소문자·숫자·`_`·`-` 3~32자. **예약어(`root`, `admin`, `ubuntu`, `dept-head` 등)는 거부**됩니다.
- 비밀번호: 10자 이상, 아이디 글자 포함 금지, 흔한 비밀번호 금지.
- 오류 문구가 **한글이라 서버 콘솔에서는 깨져 보입니다** → 위 규칙을 하나씩 확인하세요. 서버 콘솔에서는 이름을 영어로 씁니다.

실행하면 `otpauth://…` 주소가 **한 번만** 출력됩니다. 그 안의 비밀 키 부분(영문 대문자와 숫자 2~7 로 된 32글자)을 인증 앱의 **키 직접 입력**에 넣으세요(유형: 시간 기준). 이 키 문자에는 `0`·`1`·`8`·`9` 가 없습니다.

In [ ]:
!sudo k3s kubectl -n ssm exec -it deploy/ssm-api -- python -m ssm.cli create-dept-head --username [DEPT_ID] --name "[DEPT_NAME_EN]"

> 학과장은 서버 관리자를 승인하는 최상위 권한입니다. **만드는 사람이 비밀번호와 인증 키를 알고 있으면 학과장 대신 승인할 수 있으므로**, 실제 운영에서는 학과장 본인이 직접 비밀번호를 정하고 키를 등록하게 하세요.

### 인증 코드는 어디서 보나
휴대폰 인증 앱(Google/Microsoft Authenticator)의 **6자리 숫자**이고 30초마다 바뀝니다. 앱에서 **+ → 설정 키 입력 → 키 붙여넣기(유형: 시간 기준)** 로 한 번 등록해야 생깁니다. 코드가 맞는데 거절되면 폰의 날짜·시간을 **자동**으로 맞추세요.

### 로그인 확인 (🌐 사이트)
1. `[FUNNEL_URL]` → 학과장 아이디·비밀번호 → 인증 코드 칸에 6자리 → **관리자 승인** 화면이 열리면 성공.
2. 로그인 화면에서 비밀번호가 틀렸다는 메시지가 나오면 **표시** 버튼으로 입력을 확인하세요. 한/영 상태나 Caps Lock 이 흔한 원인입니다.
3. **비밀번호는 볼 수 없습니다**(되돌릴 수 없는 해시로만 저장). 잊었으면 계정을 지우고 다시 만듭니다:

In [ ]:
!sudo k3s kubectl -n ssm exec postgres-mgmt-0 -- psql -U ssm -d ssm -c "delete from users where linux_username='[DEPT_ID]'"
# 지운 뒤 위의 create-dept-head 를 다시 실행하고, 인증 앱의 옛 항목을 지운 뒤 새 키로 다시 등록

## B. 서버 관리자 가입 → 승인 (🌐 사이트)

1. 로그인 화면 → **가입하기** → **서버 관리자** 탭 → 이름, 교번/학번, 가입 사유, **리눅스 아이디**(서버에 만들어질 계정; `se` 같은 2글자나 예약어는 안 됨), 비밀번호, **비밀번호 확인**.
2. 가입하면 인증 앱 **QR 이 한 번만** 나옵니다 → 폰 카메라로 스캔(닫으면 다시 볼 수 없음).
3. 승인 전에는 로그인되지 않습니다("학과장 승인 대기").
4. **학과장 계정**으로 로그인 → **관리자 승인** → 대기 목록에서 승인(비밀번호와 인증 코드를 다시 입력).
5. 승인되면 이미 등록된 모든 서버에 관리자 리눅스 계정(sudo)이 만들어지고, **새로 등록하는 서버에는 가입할 때 자동으로** 만들어집니다.

## C. 학생 가입 (🌐 사이트)
- **학생 팀** 탭 → 팀 이름, 대표자 이름·학번, 팀원 학번(선택), 리눅스 아이디, 비밀번호·확인. 가입 즉시 로그인됩니다.
- 이 비밀번호는 **사이트와 서버 로그인에 같이** 쓰입니다. 서버에는 되돌릴 수 없는 해시로만 전달되고, 평문은 어디에도 저장되지 않습니다.
- 서버가 등록되기 전에는 "비어 있는 서버가 없습니다"가 정상입니다.

## ✅ 완료 확인

- [ ] 학과장 로그인(비밀번호+6자리 코드)이 된다
- [ ] 관리자 가입 → 학과장 승인 → 관리자 로그인이 된다
- [ ] 학생 가입 후 서버 선택 화면이 열린다
- [ ] 가입 화면에 비밀번호 확인 칸이 있다